# Phase 1 integration and teammate handover
The default run is a **bounded 10,000-trip SAMPLE**. It does not establish full-month coverage.
The normal runner executes this notebook again in a fresh kernel with `ADS_PHASE1_MODE=available`
only after tests, Ruff and the bounded run pass. January cleaning is reused, not repeated.

Core cleaning flags are immutable. UTC instants, fixed EST representations, and New York civil
calendar fields are distinct. Ambiguous/nonexistent civil times stay in unresolved buckets.
Chronological periods are Jan-Aug train, Sep-Oct validation, Nov-Dec test, January 2025 drift.
Missing source coverage never becomes zero demand; realised weather/outcomes are descriptive.
Last successful full Phase 1 execution: 15 November 2026 IST; finalized version `finalized-20261114T190620-6b22f8cb`. See shared handover for verification and reuse.


In [1]:
# Phase 1: locate the repo and record the execution environment.
import os
import sys
from pathlib import Path
ROOT = Path.cwd()
if not (ROOT / "AGENTS.md").exists():
    ROOT = ROOT.parent
if not (ROOT / "AGENTS.md").exists():
    raise RuntimeError("Run from the repository root or notebooks folder")
sys.path.insert(0, str(ROOT))
from notebooks.run_phase1 import runtime_versions, run_preflight  # noqa: E402
print(runtime_versions())
if os.environ.get("ADS_PREFLIGHT_COMPLETE") != "1":
    print("Validation receipt:", run_preflight(ROOT))
MODE = os.environ.get("ADS_PHASE1_MODE", "sample")
if MODE not in ["sample", "available", "resume", "finalize"]:
    raise ValueError("ADS_PHASE1_MODE must be sample, available, resume or finalize")

if MODE == "available":
    print("Final output-contract validation receipt:", run_preflight(ROOT))

{'python': '3.11.9', 'java': 'openjdk version "21.0.11" 2026-04-21', 'pyspark': '4.0.1', 'py4j': '0.10.9.9', 'pyarrow': '24.0.0', 'pyshp': '2.3.1', 'shapely': '2.1.2', 'pyproj': '3.7.2', 'ruff': '0.14.4', 'pytest': '9.1.1', 'nbclient': '0.10.4'}


## Reuse, monthly execution and integration
The runner checks compatible completed cleaning audits and physical footer totals before reuse.
New months run sequentially, with fresh output directories and restartable month partials.
Only narrow OD fields are cached on disk across months. Exact training medians use pooled trips,
exclude outcomes crossing the training cutoff, and never average monthly medians.

Each event reconciles to a resolved hour, an unresolved bucket or an out-of-domain count.
Reference joins assert unique keys and row preservation. Monetary denominators are retained.
Supporting-source failures become explicit blockers rather than fake observations.

In [2]:
import json
from src.phase1_sources import expected_sources, inventory_sources
from notebooks.run_phase1 import run_integration
inventory_path = os.environ.get("ADS_PHASE1_INVENTORY_PATH")
inventory = json.loads(Path(inventory_path).read_text()) if inventory_path else inventory_sources(expected_sources(ROOT))
if MODE == "available":
    contract_sample = run_integration(ROOT, inventory, sample=True)
    assert contract_sample["status"] == "sample_only"
if MODE == "finalize":
    from notebooks.run_phase1 import finalize_reference_integration
    report = finalize_reference_integration(ROOT, inventory, os.environ["ADS_PHASE1_FINALIZE_MANIFEST"])
else:
    report = run_integration(ROOT, inventory, sample=MODE == "sample")
print("Measured status:", report["status"])
print("Actual processed source months:", report["processed_source_months"])
print("Coverage:", report["coverage"])
print("Blockers:", report["blockers"])
print("Handover:", report["handover_path"])

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/11/14 19:06:21 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


26/11/14 19:06:30 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/11/14 19:06:30 WARN TaskSetManager: Stage 12 contains a task of very large size (4271 KiB). The maximum recommended task size is 1000 KiB.


26/11/14 19:06:30 WARN TaskSetManager: Stage 13 contains a task of very large size (4271 KiB). The maximum recommended task size is 1000 KiB.
26/11/14 19:06:31 WARN TaskSetManager: Stage 16 contains a task of very large size (4271 KiB). The maximum recommended task size is 1000 KiB.


26/11/14 19:06:31 WARN TaskSetManager: Stage 18 contains a task of very large size (4271 KiB). The maximum recommended task size is 1000 KiB.


Join ['hour_start_utc']: 2505864 -> 2505864


... (2 outputs left out)

Join ['event_date_nyc']: 2505864 -> 2505864


Handover: /home/mrveer/Projects/ADS/ADS-taxifleet/data/handover/finalized-20261114T190620-6b22f8cb


Reference finalization: verified_phase1 {'zone_hours': 2505864, 'request_coverage_zone_hours': 2505338, 'primary_weather_zone_hours': 2501393, 'verified_holiday_zone_hours': 2505864, 'valid_temperature_c_zone_hours': 2493766, 'valid_precipitation_previous_hour_mm_zone_hours': 2351220, 'valid_wind_metres_per_second_zone_hours': 2250491, 'valid_visibility_metres_zone_hours': 2498237}


Measured status: verified_phase1
Actual processed source months: ['2023-12', '2024-01', '2024-02', '2024-03', '2024-04', '2024-05', '2024-06', '2024-07', '2024-08', '2024-09', '2024-10', '2024-11', '2024-12', '2025-01', '2025-02']
Coverage: {'zone_hours': 2505864, 'request_coverage_zone_hours': 2505338, 'primary_weather_zone_hours': 2501393, 'verified_holiday_zone_hours': 2505864, 'valid_temperature_c_zone_hours': 2493766, 'valid_precipitation_previous_hour_mm_zone_hours': 2351220, 'valid_wind_metres_per_second_zone_hours': 2250491, 'valid_visibility_metres_zone_hours': 2498237}
Blockers: []
Handover: /home/mrveer/Projects/ADS/ADS-taxifleet/data/handover/finalized-20261114T190620-6b22f8cb


## Saved-output verification
Read-back totals and schemas are checked at every saved stage. The completed manifest contains
input provenance, versions, eligibility/time reconciliations, exact paths and coverage. `samples/`
contains at most 1,000 actual output rows per table and is not full-period data.
The data dictionary labels static/calendar attributes, derived outcomes and quality metadata.

Weather observation/publication times differ. Publication latency is unverified; contemporaneous
realised weather is not a known future predictor. Provider exclusion rates are retained as a
limitation with both recorded request-hour and pickup-hour diagnostics. No repairs are guessed.

In [3]:
from pathlib import Path
from notebooks.run_phase1 import parquet_rows
for name, table in report["tables"].items():
    assert Path(table["path"]).is_dir()
    assert parquet_rows(table["path"]) == table["rows"]
assert Path(report["handover_path"], "data_dictionary.json").is_file()
print("Saved-output footer checks passed; notebook reached its final cell.")
print("Phase 1 complete:", report["status"] == "verified_phase1")

Saved-output footer checks passed; notebook reached its final cell.
Phase 1 complete: True
